# 01 - Converter CSV para Delta Lake

Este notebook lê os arquivos CSV do bucket `landing-zone` no MinIO e grava tabelas Delta no bucket `bronze`.

In [ ]:
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("csv-to-delta") \
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.2.0,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.554") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

spark = builder.getOrCreate()

26/05/06 19:41:30 WARN Utils: Your hostname, Bonga resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 19:41:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/vanes_/spark-delta-minio-study/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vanes_/.ivy2/cache
The jars for the packages stored in: /home/vanes_/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dd47ee4d-ff83-4f24-9342-94b52e8c3a62;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.554 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (38076ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar

In [2]:
tables = ["departamentos", "funcionarios", "projetos", "alocacoes"]

for table in tables:
    csv_path = f"s3a://landing-zone/{table}.csv"
    delta_path = f"s3a://bronze/{table}"

    print(f"Lendo {csv_path}")
    df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

    df.write.format("delta").mode("overwrite").save(delta_path)

    print(f"Tabela Delta gravada em {delta_path}")

Lendo s3a://landing-zone/departamentos.csv


Tabela Delta gravada em s3a://bronze/departamentos
Lendo s3a://landing-zone/funcionarios.csv
Tabela Delta gravada em s3a://bronze/funcionarios
Lendo s3a://landing-zone/projetos.csv
Tabela Delta gravada em s3a://bronze/projetos
Lendo s3a://landing-zone/alocacoes.csv
Tabela Delta gravada em s3a://bronze/alocacoes


In [3]:
for table in tables:
    delta_path = f"s3a://bronze/{table}"
    print(f"Preview de {table}")
    spark.read.format("delta").load(delta_path).show(5, False)

Preview de departamentos
+---+------+-------------+
|id |nome  |localizacao  |
+---+------+-------------+
|1  |Vendas|São Paulo    |
|2  |TI    |Florianópolis|
+---+------+-------------+

Preview de funcionarios
+---+-----+------------------+-------+---------------+
|id |nome |cargo             |salario|departamento_id|
+---+-----+------------------+-------+---------------+
|1  |Ana  |Analista          |4500.0 |1              |
|2  |João |Desenvolvedor     |5200.0 |2              |
|3  |Carla|Técnico de Suporte|3800.0 |2              |
+---+-----+------------------+-------+---------------+

Preview de projetos
+---+----------+------------+---------------+
|id |nome      |status      |departamento_id|
+---+----------+------------+---------------+
|1  |E-commerce|em_andamento|2              |
|2  |CRM       |concluido   |1              |
+---+----------+------------+---------------+

Preview de alocacoes
+---+--------------+----------+--------------+
|id |funcionario_id|projeto_id|horas_

In [4]:
spark.stop()